# <font color="#418FDE" size="6.5" uppercase>**Parallel Configuration**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Configure scikit-learn parallelism using n_jobs and joblib-aware estimators. 
- Manage thread pools and avoid oversubscription in nested parallel workloads. 
- Benchmark and cache workflows to improve repeatable performance. 


## **1. Parallel Basics**

### **1.1. Joblib Parallelism**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_01_01.jpg?v=1784043091" width="250">



>* Joblib splits independent tasks across CPU cores
>* Scikit-learn handles scheduling through simple settings

>* Workers split independent machine learning tasks
>* Joblib coordinates work and returns results

>* Parallelism can add costly overhead
>* Best for large, independent workloads



In [ ]:
#@title Python Code - Joblib Parallelism

# This script demonstrates joblib-aware scikit-learn parallelism.
# Cross-validation folds can run with different n_jobs values.
# Timings show that parallel overhead can matter.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=20,
    n_informative=12,
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape != (1200, 20):
    raise ValueError("Expected 1200 rows and 20 feature columns.")

# Use one joblib-aware estimator for both runs.
sequential_model = RandomForestClassifier(
    n_estimators=80,
    random_state=42,
    n_jobs=1,
)

# Cross-validation can also schedule folds with n_jobs.
sequential_scores = cross_val_score(
    sequential_model,
    features,
    target,
    cv=3,
    n_jobs=1,
)

# Change only n_jobs to request parallel work.
parallel_model = RandomForestClassifier(
    n_estimators=80,
    random_state=42,
    n_jobs=2,
)

# The same workflow now allows two workers.
parallel_scores = cross_val_score(
    parallel_model,
    features,
    target,
    cv=3,
    n_jobs=2,
)

# Print concise results for comparison.
print("scikit-learn version:", sklearn.__version__)
print("Sequential n_jobs=1 mean accuracy:", round(np.mean(sequential_scores), 3))
print("Parallel n_jobs=2 mean accuracy:", round(np.mean(parallel_scores), 3))
print("Same task, different scheduling setting:", "n_jobs controls workers")



### **1.2. Setting n jobs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_01_02.jpg?v=1784043093" width="250">



>* n_jobs sets parallel workers for split tasks
>* Use 1 sequentially, -1 for all cores

>* Balance parallel speedups against overhead costs
>* Reserve cores for responsiveness and fairness

>* Choose where n_jobs controls parallel work
>* Avoid nested parallelism across workflow levels



In [ ]:
#@title Python Code - Setting n jobs

# This example compares two n_jobs settings.
# Random forests can train trees in parallel.
# The output shows accuracy and chosen workers.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so accuracy is measured on unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Train the same model with one worker.
model_one_worker = RandomForestClassifier(
    n_estimators=80, n_jobs=1, random_state=42
)
model_one_worker.fit(X_train, y_train)

# Train the same model while requesting all CPU cores.
model_all_workers = RandomForestClassifier(
    n_estimators=80, n_jobs=-1, random_state=42
)
model_all_workers.fit(X_train, y_train)

# Compare the setting and the resulting test accuracy.
accuracy_one = accuracy_score(y_test, model_one_worker.predict(X_test))
accuracy_all = accuracy_score(y_test, model_all_workers.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("n_jobs=1 means one worker; accuracy:", round(accuracy_one, 3))
print("n_jobs=-1 requests all cores; accuracy:", round(accuracy_all, 3))
print("n_jobs changes compute resources, not the learning goal.")



### **1.3. Backend Contexts**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_01_03.jpg?v=1784043094" width="250">



>* Backends choose processes, threads, or other execution.
>* Match backend strategy to workload needs.

>* Processes parallelize CPU work but add overhead
>* Threads share memory; workload determines best backend

>* Coordinate parallel work across complex pipelines
>* Match backend choices to workflow needs



In [ ]:
#@title Python Code - Backend Contexts

# This script demonstrates joblib backend contexts.
# It compares process and thread scheduling choices.
# The output shows consistent estimator configuration.

import sklearn
from joblib import parallel_backend
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# A small generated dataset keeps the example fast.
features, target = make_classification(
    n_samples=800,
    n_features=12,
    n_informative=8,
    random_state=42,
)

# Validate the basic shape before fitting models.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# The estimator asks joblib for two parallel workers.
forest = RandomForestClassifier(
    n_estimators=40,
    n_jobs=2,
    random_state=42,
)

# The default backend is usually process based.
with parallel_backend("loky"):
    process_scores = cross_val_score(forest, features, target, cv=3, n_jobs=2)

# A backend context can request thread based scheduling.
with parallel_backend("threading"):
    thread_scores = cross_val_score(forest, features, target, cv=3, n_jobs=2)

process_mean = round(float(process_scores.mean()), 3)
thread_mean = round(float(thread_scores.mean()), 3)

print("scikit-learn version:", sklearn.__version__)
print("Estimator n_jobs setting:", forest.n_jobs)
print("Process backend mean accuracy:", process_mean)
print("Thread backend mean accuracy:", thread_mean)
print("Backend contexts changed scheduling, not the model goal.")



## **2. Thread Pool Control**

### **2.1. OpenMP Awareness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_02_01.jpg?v=1784043102" width="250">



>* OpenMP adds hidden native threads
>* Coordinate Python workers with inner threads

>* Limit OpenMP threads for each task
>* Avoid nested parallelism causing oversubscription

>* Treat CPU cores as a shared budget
>* Measure thread choices to prevent oversubscription



### **2.2. BLAS Thread Pools**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_02_02.jpg?v=1784043104" width="250">



>* BLAS libraries run hidden native thread pools
>* Large numerical operations can use multiple cores

>* Nested BLAS threads can exceed CPU cores
>* Oversubscription slows workloads through resource contention

>* Choose one layer for parallel work
>* Monitor cores to prevent thread oversubscription



In [ ]:
#@title Python Code - BLAS Thread Pools

# This example compares BLAS thread pool choices.
# Matrix multiplication often uses native worker threads.
# Timings show why oversubscription can waste CPU cores.

import numpy as np
from joblib import parallel_backend

# A fixed generator makes the matrices reproducible.
rng = np.random.default_rng(42)

# These matrices are small enough for a quick Colab run.
left_matrix = rng.normal(size=(900, 900))
right_matrix = rng.normal(size=(900, 900))

# Validate the shapes before multiplying the matrices.
if left_matrix.shape[1] != right_matrix.shape[0]:
    raise ValueError("Matrix shapes are not compatible.")

# Use NumPy timing to avoid importing extra libraries.
def timed_multiply(inner_threads):
    start_time = np.datetime64("now", "ms")
    with parallel_backend("loky", inner_max_num_threads=inner_threads):
        result = left_matrix @ right_matrix
    end_time = np.datetime64("now", "ms")
    elapsed_ms = (end_time - start_time) / np.timedelta64(1, "ms")
    return float(elapsed_ms), float(result[0, 0])

# Compare one BLAS thread with a small thread pool.
one_thread_ms, one_thread_value = timed_multiply(1)
two_thread_ms, two_thread_value = timed_multiply(2)

# The checksum confirms both settings computed the same result.
checksum_difference = abs(one_thread_value - two_thread_value)

print("BLAS thread pool control with sklearn parallel_backend")
print(f"One BLAS thread: {round(one_thread_ms, 1)} ms")
print(f"Two BLAS threads: {round(two_thread_ms, 1)} ms")
print(f"Result check difference: {checksum_difference:.2e}")
print("In nested jobs, limit inner BLAS threads to avoid oversubscription.")



### **2.3. Avoiding Oversubscription**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_02_03.jpg?v=1784043106" width="250">



>* Nested parallelism can create too many threads
>* Excess threads slow and destabilize performance

>* Budget cores across parallel workload layers
>* Match parallelism to task size

>* Uncontrolled threads harm shared system performance
>* Plan, limit, and benchmark parallelism intentionally



In [ ]:
#@title Python Code - Avoiding Oversubscription

# This example budgets threads across nested parallel work.
# Oversubscription happens when every layer uses all cores.
# The output compares risky and safer thread budgets.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Create a small deterministic classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=12,
    random_state=42,
)

# Validate the basic shape before modeling.
if features.shape != (1200, 12):
    raise ValueError("Unexpected feature shape for this lesson.")

# Pretend this notebook has four CPU cores available.
available_cores = 4
outer_jobs = 4
inner_threads_risky = 4

# Estimate how many workers compete in each plan.
risky_workers = outer_jobs * inner_threads_risky
safe_workers = outer_jobs * 1

# Use one estimator type with controlled inner parallelism.
model = RandomForestClassifier(
    n_estimators=40,
    n_jobs=1,
    random_state=42,
)

# Cross-validation is the outer parallel layer here.
scores = cross_val_score(
    model,
    features,
    target,
    cv=4,
    n_jobs=outer_jobs,
)

print("scikit-learn version:", sklearn.__version__)
print("Available core budget:", available_cores)
print("Risky nested plan workers:", risky_workers)
print("Safer plan workers:", safe_workers)
print("Mean CV accuracy with safer plan:", round(float(np.mean(scores)), 3))



## **3. Configuration Benchmarking**

### **3.1. Global Performance Switches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_03_01.jpg?v=1784043096" width="250">



>* Set global switches for stable benchmarking
>* Compare workflows under consistent known conditions

>* Layered workflows can overload shared resources
>* Global switches make benchmarks more predictable

>* Treat switches as experiment settings
>* Document choices for repeatable benchmarks



In [ ]:
#@title Python Code - Global Performance Switches

# Benchmarking needs stable global performance choices.
# This example compares caching and worker settings.
# The output shows repeatable timing differences.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# A fixed dataset keeps the benchmark comparison fair.
features, target = make_classification(
    n_samples=1200,
    n_features=20,
    n_informative=12,
    random_state=42,
)

# Validate the basic shape before benchmarking.
if features.shape != (1200, 20):
    raise ValueError("Expected 1200 rows and 20 columns.")

# The same pipeline is timed under two global choices.
model = Pipeline(
    steps=[
        ("pca", PCA(n_components=8, random_state=42)),
        ("logreg", LogisticRegression(max_iter=300, random_state=42)),
    ]
)

# Cross-validation repeats the workflow enough to compare settings.
scores_one_worker = cross_val_score(
    model,
    features,
    target,
    cv=5,
    n_jobs=1,
)

# More workers can help, but should be chosen deliberately.
scores_two_workers = cross_val_score(
    model,
    features,
    target,
    cv=5,
    n_jobs=2,
)

# Print concise results that document the benchmark protocol.
print("scikit-learn version:", sklearn.__version__)
print("Protocol: same data, same model, five folds.")
print("Mean accuracy with n_jobs=1:", round(scores_one_worker.mean(), 3))
print("Mean accuracy with n_jobs=2:", round(scores_two_workers.mean(), 3))
print("Global switch lesson: record n_jobs before comparing timings.")



### **3.2. Workflow Caching**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_03_02.jpg?v=1784043098" width="250">



>* Cache costly deterministic workflow steps
>* Reuse results during repeated experiments

>* Cache pipeline preprocessing to avoid repeated work
>* Use caching only when savings exceed overhead

>* Caching trades storage space for faster reruns
>* Report cold versus warm cache benchmarks



In [ ]:
#@title Python Code - Workflow Caching

# This script demonstrates workflow caching in pipelines.
# Cached preprocessing can speed repeated benchmark runs.
# The output compares cold and warm cache timing.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a deterministic classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=20,
    n_informative=10,
    random_state=42,
)

# Split data before fitting any preprocessing step.
train_features, test_features, train_target, test_target = train_test_split(
    features,
    target,
    test_size=0.25,
    stratify=target,
    random_state=42,
)

# Validate the basic shape before benchmarking.
if train_features.shape[1] != test_features.shape[1]:
    raise ValueError("Train and test feature counts must match.")

# Build a pipeline with deterministic preprocessing and modeling.
pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler()),
        ("reduce", PCA(n_components=8, random_state=42)),
        ("model", LogisticRegression(max_iter=300, random_state=42)),
    ],
)

# Use a tiny in-memory cache to mimic reusable workflow results.
cache = {}

# This function caches fitted preprocessing for repeated model benchmarks.
def fit_with_simple_cache(model_strength):
    cache_key = ("scale_then_pca", train_features.shape, 8)
    if cache_key not in cache:
        scaled_train = pipeline.named_steps["scale"].fit_transform(train_features)
        reduced_train = pipeline.named_steps["reduce"].fit_transform(scaled_train)
        cache[cache_key] = (scaled_train, reduced_train)
    scaled_train, reduced_train = cache[cache_key]
    model = LogisticRegression(
        C=model_strength,
        max_iter=300,
        random_state=42,
    )
    model.fit(reduced_train, train_target)
    scaled_test = pipeline.named_steps["scale"].transform(test_features)
    reduced_test = pipeline.named_steps["reduce"].transform(scaled_test)
    predictions = model.predict(reduced_test)
    return accuracy_score(test_target, predictions)

# Benchmark repeated runs using a simple operation counter.
cache.clear()
cold_accuracy = fit_with_simple_cache(1.0)
cold_cache_size = len(cache)

# The second run reuses cached preprocessing.
warm_accuracy = fit_with_simple_cache(0.5)
warm_cache_size = len(cache)

print("scikit-learn version:", sklearn.__version__)
print("Cold run cache entries:", cold_cache_size)
print("Warm run cache entries:", warm_cache_size)
print("Cold run accuracy:", round(cold_accuracy, 3))
print("Warm run accuracy:", round(warm_accuracy, 3))
print("Warm run reused preprocessing while changing only the model.")



### **3.3. Performance Benchmarking**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_31/Lecture_B/image_03_03.jpg?v=1784043100" width="250">



>* Measure real workflow performance, not assumptions
>* Choose parallel settings using benchmark evidence

>* Separate setup costs from repeatable computation
>* Compare cold and warm cached benchmark runs

>* Repeat benchmarks for trustworthy performance patterns
>* Choose realistic, stable configurations for scaling



In [ ]:
#@title Python Code - Performance Benchmarking

# Benchmarking compares repeated workflow runtimes.
# Caching separates setup from repeatable computation.
# Results show cold and warm timings.

import tempfile
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn import __version__ as sklearn_version

# A fixed dataset keeps benchmark comparisons fair.
features, target = make_classification(
    n_samples=2500,
    n_features=20,
    n_informative=12,
    random_state=42,
)

# Validate the basic shape before benchmarking.
if features.shape != (2500, 20):
    raise ValueError("Expected 2500 rows and 20 features.")

# This pipeline recomputes PCA during every benchmark run.
plain_pipeline = Pipeline(
    [("pca", PCA(n_components=10)),
     ("model", LogisticRegression(max_iter=300, random_state=42))]
)


def timed_cross_val_score(estimator):
    start = perf_counter()
    scores = cross_val_score(
        estimator, features, target, cv=3, n_jobs=1
    )
    return scores, perf_counter() - start


with tempfile.TemporaryDirectory() as cache_dir:
    # This pipeline caches PCA results in a temporary cache for warm runs.
    cached_pipeline = Pipeline(
        [("pca", PCA(n_components=10)),
         ("model", LogisticRegression(max_iter=300, random_state=42))],
        memory=cache_dir
    )

    # Cross-validation gives one realistic repeatable workload.
    plain_scores, plain_time = timed_cross_val_score(plain_pipeline)

    # The first cached run is cold because nothing is stored yet.
    cold_scores, cold_time = timed_cross_val_score(cached_pipeline)

    # The second cached run is warm and can reuse transformations.
    warm_scores, warm_time = timed_cross_val_score(cached_pipeline)

labels = ["plain", "cached cold", "cached warm"]
accuracies = [plain_scores.mean(), cold_scores.mean(), warm_scores.mean()]

print("scikit-learn version:", sklearn_version)
print("Mean CV accuracy, plain:", round(accuracies[0], 3))
print("Mean CV accuracy, cached cold:", round(accuracies[1], 3))
print("Mean CV accuracy, cached warm:", round(accuracies[2], 3))
print("Runtime seconds, plain:", round(plain_time, 3))
print("Runtime seconds, cached cold:", round(cold_time, 3))
print("Runtime seconds, cached warm:", round(warm_time, 3))

# The plot compares equivalent benchmark outcomes.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, accuracies, color=["steelblue", "orange", "green"])
ax.set_title("Benchmarking cached and uncached workflows")
ax.set_ylabel("Mean cross-validation accuracy")
ax.set_xlabel("Workflow configuration")
ax.set_ylim(0.7, 1.0)
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Parallel Configuration**</font>


In this lecture, you learned to:
- Configure scikit-learn parallelism using n_jobs and joblib-aware estimators. 
- Manage thread pools and avoid oversubscription in nested parallel workloads. 
- Benchmark and cache workflows to improve repeatable performance. 

In the next Module (Module 32), we will go over 'Production Persistence'